# Basic Mixture of Experts — capacity-matched Colab comparison

This notebook trains `moe_basic`, the native MoE comparator for the dataset-per-expert methods. It uses the same shared encoder, expert count, expert MLP dimensions, gate, class vocabulary, optimizer, routing implementation, and evaluation pipeline. The controlled difference is that experts are generic: no expert gets a dataset assignment, Stage B performs no dataset-specific warm-start, the router receives no dataset-ID loss, and all experts learn jointly from the pooled task objective.

Before running, select a GPU runtime, add a Colab Secret named `GITHUB_TOKEN`, and place the selected NF-v3 files under `MyDrive/NIDS_datasets/`. Use the same dataset list and hyperparameters as the dataset-MoE run you want to compare.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = "selimsidan"
GITHUB_REPO = "dataset_moe_nids"
GITHUB_BRANCH = "main"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"

DRIVE_DATA_DIR = "/content/drive/MyDrive/NIDS_datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs"
EXECUTION_MODE = "out_of_core_full"  # out_of_core_full | in_memory_smoke
ACTIVE_DATASETS = [
    "NF-UNSW-NB15-v3",
    "NF-ToN-IoT-v3",
    "NF-BoT-IoT-v3",
    "NF-CICIDS2018-v3",
]

RUN_NAME = "nfv3_4way_moe_basic_seed0_v1"
SEED = 0
LATENT_DIM = 64
ENCODER_HIDDEN_DIMS = [128]
EXPERT_HIDDEN_DIMS = []
DROPOUT = 0.2
ROUTING = "dense"  # dense matches the soft dataset-MoE comparison
LAMBDA_BALANCE = 0.1
EPOCHS_A = 30
EPOCHS_C = 30
BATCH_SIZE = 512
FORCE_RESTART = False
RUN_TESTS = True
# ==================================================================

## 1. Secure checkout and environment setup

The GitHub token is passed through a temporary HTTP header and is not stored in the clone URL or Git configuration.

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("This notebook is intended for Google Colab.") from exc
drive.mount("/content/drive")
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f"Add {GITHUB_SECRET_NAME} in Colab Secrets and grant notebook access.")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ | {
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
}
repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
repo_dir = Path("/content") / GITHUB_REPO
if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", repo_url, str(repo_dir)], env=git_env, check=True)
git_env.clear()
token = auth = None
os.chdir(repo_dir)
os.environ["NIDS_DRIVE_BASE"] = DRIVE_DATA_DIR
os.environ["NIDS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
os.environ["NIDS_SCRATCH_DIR"] = "/content/dataset_moe_nids_scratch"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Repository:", repo_dir)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Preflight and resolved basic-MoE contract

The assertions below prevent an accidental dataset-supervised or dataset-owned run. They also verify that the basic and dataset MoEs have identical parameter counts for the selected capacity settings.

In [ ]:
import torch
from data.registry import get_spec
from models.encoder import SharedEncoder
from training.config import load_config
from training.model_utils import build_model
if EXECUTION_MODE not in {"out_of_core_full", "in_memory_smoke"}:
    raise ValueError("EXECUTION_MODE must be out_of_core_full or in_memory_smoke")
if ROUTING not in {"dense", "top1"}:
    raise ValueError("ROUTING must be dense or top1")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU and reconnect.")
if len(ACTIVE_DATASETS) != len(set(ACTIVE_DATASETS)):
    raise ValueError("ACTIVE_DATASETS contains duplicates")
if EXECUTION_MODE == "out_of_core_full" and not 2 <= len(ACTIVE_DATASETS) <= 4:
    raise ValueError("Full-data mode requires two, three, or four datasets")
missing = {}
for name in ACTIVE_DATASETS:
    spec = get_spec(name)
    if not any(Path(path).is_file() for path in spec.paths):
        missing[name] = spec.paths
if missing:
    raise FileNotFoundError("Missing selected datasets:\n" + "\n".join(f"  {name}: {paths}" for name, paths in missing.items()))
if EXECUTION_MODE == "out_of_core_full":
    specs = [get_spec(name) for name in ACTIVE_DATASETS]
    aliases = [spec.feature_alias for spec in specs]
    if any(spec.kind != "file" for spec in specs) or any(value != aliases[0] for value in aliases[1:]):
        raise ValueError("Full-data combinations must use schema-compatible single-file NF-v3 datasets")
    assert len(aliases[0]) == 47
contract_overrides = [
    "architecture=moe_basic",
    "training.stage_b.warmstart_mode=random_init",
    "training.stage_c.gate_supervision=none",
    "training.stage_c.expert_update_policy=all",
    "training.stage_c.lambda_expert_anchor=0.0",
    f"model.gate.routing={ROUTING}",
]
preview = load_config("config/default.yaml", contract_overrides)
stage_b = preview["training"]["stage_b"]
stage_c = preview["training"]["stage_c"]
assert preview["architecture"] == "moe_basic"
assert stage_b["warmstart_mode"] == "random_init"
assert stage_c["gate_supervision"] == "none"
assert stage_c["expert_update_policy"] == "all"
assert stage_c["lambda_expert_anchor"] == 0.0
model_cfg = preview["model"].copy()
model_cfg["latent_dim"] = LATENT_DIM
model_cfg["encoder"] = model_cfg["encoder"] | {"hidden_dims": ENCODER_HIDDEN_DIMS, "dropout": DROPOUT}
model_cfg["expert"] = model_cfg["expert"] | {"hidden_dims": EXPERT_HIDDEN_DIMS, "dropout": DROPOUT}
model_cfg["gate"] = model_cfg["gate"] | {"routing": ROUTING}
input_dim = 47 if EXECUTION_MODE == "out_of_core_full" else 8
classes = ["Benign", "Attack"]
basic = build_model("moe_basic", SharedEncoder(input_dim, ENCODER_HIDDEN_DIMS, LATENT_DIM, dropout=DROPOUT), ACTIVE_DATASETS, classes, model_cfg)
dataset_moe = build_model("moe_dataset_soft", SharedEncoder(input_dim, ENCODER_HIDDEN_DIMS, LATENT_DIM, dropout=DROPOUT), ACTIVE_DATASETS, classes, model_cfg)
basic_params = sum(p.numel() for p in basic.parameters())
dataset_params = sum(p.numel() for p in dataset_moe.parameters())
assert basic_params == dataset_params
print("GPU:", torch.cuda.get_device_name(0))
print("Datasets:", ACTIVE_DATASETS)
print("Generic experts:", basic.expert_names)
print("Parameter parity:", basic_params, "==", dataset_params)
print("Resolved contract:", stage_b["warmstart_mode"], stage_c["gate_supervision"], stage_c["expert_update_policy"])
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
else:
    print("Tests skipped by configuration.")

## 3. Train Stage A → generic initialization → joint MoE and evaluate

Stage A remains identical to the dataset-MoE. Stage B only materializes a deterministically initialized, capacity-matched expert bank. Stage C is the native pooled MoE training phase.

In [ ]:
overrides = [
    f"run_name={RUN_NAME}",
    f"seed={SEED}",
    "architecture=moe_basic",
    "training.stage_b.warmstart_mode=random_init",
    "training.stage_c.gate_supervision=none",
    "training.stage_c.expert_update_policy=all",
    "training.stage_c.lambda_expert_anchor=0.0",
    "data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]",
    "training.device=cuda",
    f"training.force_restart={str(FORCE_RESTART).lower()}",
    f"training.epochs_a={EPOCHS_A}",
    f"training.epochs_c={EPOCHS_C}",
    f"training.batch_size={BATCH_SIZE}",
    f"model.latent_dim={LATENT_DIM}",
    "model.encoder.hidden_dims=[" + ",".join(map(str, ENCODER_HIDDEN_DIMS)) + "]",
    "model.expert.hidden_dims=[" + ",".join(map(str, EXPERT_HIDDEN_DIMS)) + "]",
    f"model.encoder.dropout={DROPOUT}",
    f"model.expert.dropout={DROPOUT}",
    f"model.gate.routing={ROUTING}",
    "training.stage_c_unfreeze=all",
    f"load_balance.lambda_balance={LAMBDA_BALANCE}",
]
module = "training.ooc_run" if EXECUTION_MODE == "out_of_core_full" else "training.run"
cmd = [sys.executable, "-m", module, "--config", "config/default.yaml"]
if EXECUTION_MODE == "in_memory_smoke":
    cmd += ["--mode", "smoke"]
for override in overrides:
    cmd += ["--set", override]
print("Launching:", module)
subprocess.run(cmd, check=True, env=os.environ.copy())
print("Basic MoE training and evaluation completed.")

## 4. Review persisted results

In [ ]:
import pandas as pd
from IPython.display import display
result_dir = Path(DRIVE_OUTPUT_DIR) / "results" / RUN_NAME
csv_files = sorted(result_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No result CSVs found in {result_dir}")
for path in csv_files:
    print(f"=== {path.name} ===")
    display(pd.read_csv(path))
print("Detailed artifacts:", result_dir)
print("Checkpoints:", Path(DRIVE_OUTPUT_DIR) / "checkpoints" / RUN_NAME)

## Fair-comparison checklist

- Use the identical `ACTIVE_DATASETS`, seed, split settings, dimensions, dropout, routing, batch size, Stage A epochs, and Stage C epochs as the dataset-MoE run.
- Keep a distinct `RUN_NAME`; checkpoint contracts intentionally prevent cross-architecture reuse.
- `EPOCHS_B` is intentionally absent: dataset-specific expert warm-start is part of the treatment being removed.
- Start with `in_memory_smoke`, then use a new production run name for `out_of_core_full`.
- To return to the original method, use its existing notebook or set `architecture=moe_dataset_soft`; no old path was replaced.